In [9]:
import numpy as np
import csv

In [10]:
#========================================
#1. LEITURA DO CSV
#========================================

def load_data(filename):
    X = []
    y = []

    with open(filename, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)

        for row in reader:
            #target
            survived = row["survived"]
            if survived == "":
                continue

            y.append(int(survived))

            # Features selecionadas
            #vamos usar:
            #pclass, sex, age, sibsp, parch, fare

            pclass = float(row["pclass"]) if row["pclass"] != "" else 0

            #convertendo sexo
            sex = 1 if row["sex"] == "female" else 0

            age = float(row["age"]) if row["age"] != "" else -1

            sibsp = float(row["sibsp"]) if row["sibsp"] != "" else 0
            parch = float(row["parch"]) if row["parch"] != "" else 0
            fare = float(row["fare"]) if row["fare"] != "" else 0

            X.append([pclass, sex, age, sibsp, parch, fare])

    return np.array(X), np.array(y).reshape(-1, 1)

In [11]:
#====================================
#2. TRATAMENTO DE DADOS
#====================================

def fill_missing_age(X):
    ages = X[:, 2]
    # 1. Filtra as idades válidas (diferentes de -1) e calcula a média
    valid_ages = ages[ages != -1]
    mean_age = np.mean(valid_ages)
    
    # 2. Onde for -1, substitui pela média; caso contrário, mantém a idade original
    X[:, 2] = np.where(ages == -1, mean_age, ages)
    return X


def normalize(X):
    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0)
    return (X - mean) / (std + 1e-8)


def add_bias(X):
    ones = np.ones((X.shape[0], 1))
    return np.hstack((ones , X))

In [12]:
#===============================
# 3. FUNÇÕES DE MODELO
#===============================

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def compute_cost(x, y, beta):
    m = len(y)
    h = sigmoid(x @ beta)
    epsilon = 1e-8

    cost = -(1/m) * np.sum(y * np.log(h + epsilon) + (1 - y) * np.log(1 - h + epsilon))

    return cost

def gradient_descent(X, y, beta, Ir, epochs):
    m = len(y)

    for i in range(epochs):
        h = sigmoid(X @ beta)
        gradient = (1/m) * (X.T @ (h - y))

        beta = beta - Ir * gradient

        if i % 100 == 0:
            print(f"Epoch {i} - Cost: {compute_cost(X, y, beta):.4f}")

    return beta


In [13]:
#===============================
# 4. TREINAMENTO
#===============================

def train(X, y):
    x = fill_missing_age(X)
    x = normalize(X)
    x = add_bias(X)

    beta = np.zeros((X.shape[1], 1))

    beta = gradient_descent(X, y, beta, Ir=0.01, epochs=2000)

    return beta, X

In [14]:
#===============================
# 5. PREDIÇÃO E CALCULO DA MATRIZ DE CONFUSÃO 
#===============================

def predict(X, beta):
    probs = sigmoid(X @ beta)
    return (probs >= 0.5).astype(int)

def confusion_matrix(y_true, y_pred):
    #Achataram os arrays para garantir que tenham apenas 1 dimensão
    y_t = y_true.flatten()
    y_p = y_pred.flatten()

    # O operador '&' faz a comparação bit a bita (element-wise) nos arrays do numpy
    # Faz AND lógico, elemento por elemento de cada array.
    # Por Exemplo:
    # y_t = np.array([1, 0, 1, 1])
    # cond1 = (y_t == 1)
    # Resultado: [True, False, True, True]
    VP = np.sum((y_t == 1 ) & (y_p == 1))
    VN = np.sum((y_t == 0 ) & (y_p == 0))
    FP = np.sum((y_t == 0 ) & (y_p == 1))
    FN = np.sum((y_t == 1 ) & (y_p == 0))

    return VP , VN, FP, FN

def calculate_metrics(VP, VN, FP, FN):
    acc = (VP + VN) / (VP + VN + FP + FN)
    prec = VP / (VP + FP)
    rec = VP / (VP + FN)
    spec = VN / (VN + FP)
    f1 = 2 * (prec * rec) /  (prec + rec)

    return acc, prec, rec, spec, f1 

In [15]:
#===============================
# 6. SPLIT TRERINO  / TESTE
#===============================

def train_test_split(X, y, test_size=0.2):
    np.random.seed(42)

    indices = np.arange(len(X))
    np.random.shuffle(indices)

    split = int(len(X) * (1 - test_size))

    train_idx = indices[:split]
    test_idx = indices[split:]
    
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

In [16]:
#===============================
# 7. EXECUÇÃO
#===============================

X, y = load_data("titanic.csv")

X_train, X_test, y_train, y_test = train_test_split(X, y)

beta, x_train_processed = train(X_train, y_train)

# Processar teste com MESMAS transformações
X_test_proc = fill_missing_age(X_test)
X_test_proc = normalize(X_test)
X_test_proc = add_bias(X_test)

# Vetor de Predições
y_pred = predict(X_test, beta)

# Métricas da Matriz de Confusão  
VP, VN, FP, FN = confusion_matrix(y_test, y_pred)

print("\n" + "="*30)
print(" MATRIZ DE CONFUSÃO ")
print("="*30)
print(f"Verdadeiro Positivos (VP): {VP}")
print(f"Verdadeiro Negativos (VN): {VN}")
print(f"Falsos Positivos (FP):     {FP}")
print(f"Falso Negativos (FN):      {FN}")

# 2. Calcular e exibir as Métricas
acc, prec, rec, spec, f1 = calculate_metrics(VP, VN, FP, FN)

print("\n" + "="*30)
print(" MÉTRICAS DE AVALIAÇÃO ")
print("="*30)
print(f"Acurácia:       {acc:.4f} ({(acc*100):.1f}%)")
print(f"Precisão:       {prec:.4f} ({(prec*100):.1f}%)")
print(f"Revocação:      {rec:.4f} ({(rec*100):.1f}%)")
print(f"Especificidade: {spec:.4f} ({(spec*100):.1f}%)")
print(f"F1-Score:       {f1:.4f} ({(f1*100):.1f}%)")

Epoch 0 - Cost: 0.6404
Epoch 100 - Cost: 2.1174
Epoch 200 - Cost: 1.1066
Epoch 300 - Cost: 1.1172
Epoch 400 - Cost: 2.0853
Epoch 500 - Cost: 0.9696
Epoch 600 - Cost: 1.0634
Epoch 700 - Cost: 2.0422
Epoch 800 - Cost: 0.8650
Epoch 900 - Cost: 1.0078
Epoch 1000 - Cost: 2.0124
Epoch 1100 - Cost: 0.8157
Epoch 1200 - Cost: 0.9584
Epoch 1300 - Cost: 2.0028
Epoch 1400 - Cost: 0.8090
Epoch 1500 - Cost: 0.9100
Epoch 1600 - Cost: 2.0029
Epoch 1700 - Cost: 0.8305
Epoch 1800 - Cost: 0.8597
Epoch 1900 - Cost: 2.0057

 MATRIZ DE CONFUSÃO 
Verdadeiro Positivos (VP): 19
Verdadeiro Negativos (VN): 154
Falsos Positivos (FP):     1
Falso Negativos (FN):      88

 MÉTRICAS DE AVALIAÇÃO 
Acurácia:       0.6603 (66.0%)
Precisão:       0.9500 (95.0%)
Revocação:      0.1776 (17.8%)
Especificidade: 0.9935 (99.4%)
F1-Score:       0.2992 (29.9%)
